# CyberSentinel — Model training on CICIDS2017

This notebook is the full recipe for the detection model. Run it top to bottom to reproduce
`python-engine/models/model_v1.pkl`.

What it does:

1. Loads the eight CICIDS2017 CSV files (see `DATASET.md`).
2. Cleans them and merges the original labels into six classes.
3. Compares feature groups and ways of handling class imbalance.
4. Trains the final Random Forest and evaluates it on a held-out test set.
5. Saves the model, the feature list, and a metrics report.

To train a new version, change `MODEL_VERSION` in the next cell and rerun everything.
Never overwrite an existing model file.

In [1]:
MODEL_VERSION = "v1"
SEED = 42                  # fixed so every run gives the same split and the same model
BENIGN_SAMPLE = 500_000    # benign flows kept after cleaning (the dataset has about 2 million)
TEST_SIZE = 0.20

In [2]:
import glob, json, platform, re, time
from datetime import date
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

# Works whether the notebook is run from training/ or from the repo root
TRAIN_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd() / "training"
DATA_DIR = TRAIN_DIR / "data" / "MachineLearningCVE"
REPORT_DIR = TRAIN_DIR / "reports"
MODEL_DIR = TRAIN_DIR.parent / "python-engine" / "models"
REPORT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / f"model_{MODEL_VERSION}.pkl"
assert not MODEL_PATH.exists(), f"{MODEL_PATH.name} already exists. Bump MODEL_VERSION instead of overwriting."

files = sorted(glob.glob(str(DATA_DIR / "*.csv")))
assert len(files) == 8, f"Expected 8 CSV files in {DATA_DIR}, found {len(files)}. See DATASET.md."
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.7.2 | pandas 2.3.3 | numpy 2.2.6


## 1. Features and classes

Every feature must be something the live engine can compute from packet headers and timing,
**in exactly the same way the dataset computed it**. The candidates are split into groups so
that section 4 can measure what each group adds.

| Group | Why it is a separate group |
|---|---|
| `CORE` | Packet counts, payload sizes and timing. Reproducible from live traffic. |
| `FLAGS` | TCP flag counts. In this dataset they are only ever 0 or 1 and are wrong for obvious cases (a port-scan flow shows `SYN Flag Count = 0`), so live values would not match. |
| `WIN` | Initial TCP window sizes. These largely identify the operating system and tool of the lab's attacker machines. |
| `PORT` | Destination port. A model can memorise "port 21 or 22 means brute force" instead of learning behaviour. |

In [3]:
CORE = ["Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Total Length of Fwd Packets", "Total Length of Bwd Packets",
        "Fwd Packet Length Max", "Fwd Packet Length Mean",
        "Bwd Packet Length Max", "Bwd Packet Length Mean",
        "Packet Length Mean", "Packet Length Std",
        "Flow Bytes/s", "Flow Packets/s",
        "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
        "Fwd IAT Mean", "Bwd IAT Mean", "Down/Up Ratio"]
FLAGS = ["SYN Flag Count", "FIN Flag Count", "RST Flag Count", "PSH Flag Count", "ACK Flag Count"]
WIN = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]
PORT = ["Destination Port"]
ALL_CANDIDATES = CORE + FLAGS + WIN + PORT

# Original label -> our class. Labels not listed here are dropped (see PREPROCESSING.md).
CLASS_MAP = {
    "BENIGN": "Benign",
    "DoS Hulk": "DoS", "DoS GoldenEye": "DoS", "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS",
    "DDoS": "DDoS",
    "PortScan": "PortScan",
    "FTP-Patator": "BruteForce", "SSH-Patator": "BruteForce",
    "Bot": "Botnet",
}

## 2. Load and clean

In [4]:
parts, raw_labels = [], {}
for f in files:
    # Column names in the files have leading spaces, so match on the stripped name
    d = pd.read_csv(f, encoding="latin-1", usecols=lambda c: c.strip() in ALL_CANDIDATES + ["Label"])
    d.columns = [c.strip() for c in d.columns]
    for k, v in d["Label"].value_counts().items():
        raw_labels[k] = raw_labels.get(k, 0) + int(v)
    parts.append(d)
df = pd.concat(parts, ignore_index=True)
del parts

counts = {"raw rows": len(df)}

df["label"] = df["Label"].map(CLASS_MAP)
dropped_labels = {k: v for k, v in raw_labels.items() if k not in CLASS_MAP}
df = df[df["label"].notna()].drop(columns="Label")
counts["after dropping excluded labels"] = len(df)

# Flow Bytes/s and Flow Packets/s are Infinity or NaN when the flow duration is zero
df[ALL_CANDIDATES] = df[ALL_CANDIDATES].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=ALL_CANDIDATES)
df = df[df["Flow Duration"] >= 0]          # a few rows have a negative duration
counts["after removing Infinity/NaN/negative duration"] = len(df)

# Identical rows would otherwise land in both train and test and inflate the scores
df = df.drop_duplicates()
counts["after removing exact duplicates"] = len(df)

class_counts_clean = df["label"].value_counts()
print(pd.Series(counts).to_string())
print()
print("Dropped labels:", {re.sub(r"[^ -~]+", "-", k): v for k, v in dropped_labels.items()})
print()
print(class_counts_clean.to_string())

raw rows                                         2830743
after dropping excluded labels                   2828516
after removing Infinity/NaN/negative duration    2825534
after removing exact duplicates                  2476428

Dropped labels: {'Infiltration': 36, 'Web Attack - Brute Force': 1507, 'Web Attack - XSS': 652, 'Web Attack - Sql Injection': 21, 'Heartbleed': 11}

label
Benign        2053034
DoS            193593
DDoS           128011
PortScan        90694
BruteForce       9148
Botnet           1948


## 3. Sample and split

Benign traffic is about 83% of the cleaned data. It is downsampled so that training fits
comfortably in memory and the model is not swamped by one class. All attack rows are kept.
The split is stratified, so each class has the same share in train and test.

In [5]:
benign = df[df["label"] == "Benign"].sample(n=BENIGN_SAMPLE, random_state=SEED)
data = pd.concat([benign, df[df["label"] != "Benign"]], ignore_index=True)
del df, benign

train, test = train_test_split(data, test_size=TEST_SIZE, stratify=data["label"], random_state=SEED)
CLASSES = sorted(data["label"].unique())
split_table = pd.DataFrame({"train": train["label"].value_counts(), "test": test["label"].value_counts()}).loc[CLASSES]
print(split_table.to_string())
print("\ntotal:", len(train), "train,", len(test), "test")

             train    test
label                     
Benign      400000  100000
Botnet        1559     389
BruteForce    7318    1830
DDoS        102409   25602
DoS         154874   38719
PortScan     72555   18139

total: 738715 train, 184679 test


## 4. Experiments

Each experiment trains a smaller forest (60 trees) on the same split and reports precision and
recall per class on the test set. **Macro F1** is the average F1 across classes, so a rare class
counts as much as a common one.

In [6]:
def experiment(name, features, class_weight=None, oversample_to=None):
    tr = train
    if oversample_to:   # duplicate rows of rare classes, in the training set only
        tr = pd.concat([resample(g, n_samples=oversample_to, replace=True, random_state=SEED)
                        if len(g) < oversample_to else g for _, g in train.groupby("label")])
    m = RandomForestClassifier(n_estimators=60, max_depth=20, min_samples_leaf=2, n_jobs=-1,
                               random_state=SEED, class_weight=class_weight)
    m.fit(tr[features], tr["label"])
    pred = m.predict(test[features])
    row = {"experiment": name, "n_features": len(features),
           "macro_f1": f1_score(test["label"], pred, average="macro")}
    p = precision_score(test["label"], pred, average=None, labels=CLASSES, zero_division=0)
    r = recall_score(test["label"], pred, average=None, labels=CLASSES)
    for c, pi, ri in zip(CLASSES, p, r):
        row[f"{c} P"], row[f"{c} R"] = pi, ri
    return row

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

### 4.1 Which feature groups?

In [7]:
feature_results = pd.DataFrame([
    experiment("core", CORE),
    experiment("core + flags", CORE + FLAGS),
    experiment("core + window", CORE + WIN),
    experiment("core + port", CORE + PORT),
    experiment("core + window + port", CORE + WIN + PORT),
    experiment("all candidates", ALL_CANDIDATES),
]).set_index("experiment")
feature_results.round(3)

,n_features,macro_f1,Benign P,Benign R,Botnet P,Botnet R,BruteForce P,BruteForce R,DDoS P,DDoS R,DoS P,DoS R,PortScan P,PortScan R
experiment,,,,,,,,,,,,,,
core,19,0.948,0.995,0.998,0.879,0.599,0.999,0.972,0.999,0.999,0.996,0.992,0.997,0.999
core + flags,24,0.948,0.995,0.998,0.882,0.596,0.999,0.972,0.999,0.999,0.997,0.992,0.997,0.999
core + window,21,0.983,0.998,0.999,0.903,0.910,0.999,0.993,1.000,0.999,0.999,0.997,0.997,0.999
core + port,20,0.958,0.997,0.998,0.852,0.694,0.993,0.996,1.000,0.999,0.997,0.995,0.997,0.999
core + window + port,22,0.986,0.999,0.999,0.916,0.920,1.000,0.999,1.000,0.999,0.999,0.999,0.997,0.999
all candidates,27,0.985,0.999,0.999,0.917,0.910,1.000,0.999,1.000,1.000,0.999,0.999,0.997,0.999


In [8]:
# Why the window sizes help so much: each attack class has a few signature values,
# which belong to the lab's attacker machines and tools rather than to the attack itself.
win_table = {c: data.loc[data["label"] == c, "Init_Win_bytes_forward"].value_counts(normalize=True).head(3).round(2).to_dict()
             for c in CLASSES}
for c, v in win_table.items():
    print(f"{c:11s}", v)

Benign      {-1: 0.42, 8192: 0.13, 29200: 0.1}
Botnet      {8192: 0.63, 237: 0.33, 1255: 0.01}
BruteForce  {29200: 0.76, 229: 0.22, 259: 0.02}
DDoS        {256: 0.55, 8192: 0.45, 29200: 0.0}
DoS         {0: 0.3, 251: 0.29, 274: 0.22}
PortScan    {29200: 0.5, 1024: 0.5, 251: 0.0}


**Decision: use `CORE` only.**

- `FLAGS` add nothing measurable, and the live engine cannot reproduce their values.
- `WIN` gives the biggest jump (mostly Botnet recall), but the table above shows it works by
  recognising the lab's attacker machines. On a different network those values would differ,
  so the gain is unlikely to carry over and benign machines sharing a value could be flagged.
- `PORT` gives a small gain by memorising service ports.

The cost of this choice is a weaker Botnet class. That is reported honestly in section 6 rather
than hidden behind features that only work inside this dataset.

In [9]:
SELECTED_FEATURES = CORE

### 4.2 How to handle class imbalance?

In [10]:
imbalance_results = pd.DataFrame([
    experiment("no weighting", SELECTED_FEATURES),
    experiment("class_weight=balanced", SELECTED_FEATURES, class_weight="balanced"),
    experiment("class_weight=balanced_subsample", SELECTED_FEATURES, class_weight="balanced_subsample"),
    experiment("oversample rare classes to 30k", SELECTED_FEATURES, oversample_to=30_000),
]).set_index("experiment")
imbalance_results.round(3)

,n_features,macro_f1,Benign P,Benign R,Botnet P,Botnet R,BruteForce P,BruteForce R,DDoS P,DDoS R,DoS P,DoS R,PortScan P,PortScan R
experiment,,,,,,,,,,,,,,
no weighting,19,0.948,0.995,0.998,0.879,0.599,0.999,0.972,0.999,0.999,0.996,0.992,0.997,0.999
class_weight=balanced,19,0.856,0.997,0.969,0.125,0.967,0.918,0.974,0.999,0.999,0.993,0.994,0.997,0.999
class_weight=balanced_subsample,19,0.857,0.997,0.969,0.126,0.972,0.912,0.974,0.999,0.999,0.994,0.993,0.996,0.999
oversample rare classes to 30k,19,0.886,0.996,0.986,0.219,0.871,0.997,0.974,0.999,0.999,0.996,0.992,0.997,0.999


**Decision: no weighting.** Weighting and oversampling raise Botnet recall but its precision
collapses, meaning most "Botnet" alerts would be benign traffic. For a system that can block
IPs automatically, false alarms are the more expensive mistake.

In [11]:
CLASS_WEIGHT = None

## 5. Train the final model

Preprocessing and the classifier are saved together as one pipeline, so the engine cannot
apply one without the other. The imputer only matters for live flows where a rate could not be
computed; the training data has no missing values after cleaning.

In [12]:
model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("forest", RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                      n_jobs=-1, random_state=SEED, class_weight=CLASS_WEIGHT)),
])
t0 = time.time()
model.fit(train[SELECTED_FEATURES], train["label"])
train_seconds = time.time() - t0
print(f"trained in {train_seconds:.0f} s")

trained in 33 s


## 6. Evaluate on the held-out test set

In [13]:
X_test, y_test = test[SELECTED_FEATURES], test["label"]
pred = model.predict(X_test)
report = classification_report(y_test, pred, labels=CLASSES, digits=4, output_dict=True)
print(classification_report(y_test, pred, labels=CLASSES, digits=4))
cm = confusion_matrix(y_test, pred, labels=CLASSES)
cm_df = pd.DataFrame(cm, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])
cm_df

              precision    recall  f1-score   support

      Benign     0.9945    0.9976    0.9961    100000
      Botnet     0.8788    0.5964    0.7106       389
  BruteForce     0.9989    0.9721    0.9853      1830
        DDoS     0.9994    0.9988    0.9991     25602
         DoS     0.9962    0.9920    0.9941     38719
    PortScan     0.9966    0.9989    0.9978     18139

    accuracy                         0.9956    184679
   macro avg     0.9774    0.9260    0.9472    184679
weighted avg     0.9956    0.9956    0.9955    184679



,pred Benign,pred Botnet,pred BruteForce,pred DDoS,pred DoS,pred PortScan
true Benign,99762,32,2,14,131,59
true Botnet,157,232,0,0,0,0
true BruteForce,51,0,1779,0,0,0
true DDoS,29,0,0,25572,1,0
true DoS,304,0,0,2,38411,2
true PortScan,7,0,0,0,13,18119


In [14]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=CLASSES, normalize="true"),
                       display_labels=CLASSES).plot(ax=ax, cmap="Blues", values_format=".3f", colorbar=False)
ax.set_title(f"model_{MODEL_VERSION}: share of each true class (rows sum to 1)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
cm_png = REPORT_DIR / f"model_{MODEL_VERSION}_confusion_matrix.png"
fig.savefig(cm_png, dpi=130)
plt.close(fig)
print("saved", cm_png.name)

saved model_v1_confusion_matrix.png


### 6.1 Confidence threshold for blocking

The engine only blocks automatically when the model is confident. This table shows, for each
threshold, how many attack flows would still be blocked and how many benign flows would be
blocked by mistake.

In [15]:
proba = model.predict_proba(X_test)
labels_ = model.classes_
top = labels_[proba.argmax(axis=1)]
conf = proba.max(axis=1)
is_attack = (y_test != "Benign").to_numpy()
rows = []
for t in [0.50, 0.70, 0.80, 0.90, 0.95, 0.99]:
    block = (top != "Benign") & (conf >= t)
    rows.append({"threshold": t,
                 "attack flows blocked": block[is_attack].mean(),
                 "benign flows blocked by mistake": block[~is_attack].mean(),
                 "benign mistakes per 100,000 flows": block[~is_attack].mean() * 100_000})
threshold_table = pd.DataFrame(rows).set_index("threshold")
threshold_table.round(5)

,attack flows blocked,benign flows blocked by mistake,"benign mistakes per 100,000 flows"
threshold,,,
0.50,0.99347,0.00236,236.0
0.70,0.99136,0.00150,150.0
0.80,0.98955,0.00111,111.0
0.90,0.98661,0.00083,83.0
0.95,0.98290,0.00068,68.0
0.99,0.94819,0.00046,46.0


In [16]:
importances = pd.Series(model.named_steps["forest"].feature_importances_, index=SELECTED_FEATURES).sort_values(ascending=False)
importances.round(4)

Packet Length Std              0.1074
Bwd Packet Length Mean         0.0992
Total Length of Fwd Packets    0.0987
Packet Length Mean             0.0916
Bwd Packet Length Max          0.0852
Fwd Packet Length Max          0.0848
Total Length of Bwd Packets    0.0738
Fwd Packet Length Mean         0.0676
Total Fwd Packets              0.0449
Flow Packets/s                 0.0396
Flow IAT Std                   0.0395
Flow IAT Max                   0.0363
Flow Duration                  0.0283
Flow IAT Mean                  0.0270
Fwd IAT Mean                   0.0268
Flow Bytes/s                   0.0222
Total Backward Packets         0.0190
Bwd IAT Mean                   0.0050
Down/Up Ratio                  0.0031
dtype: float64

## 7. Save the model, the feature list and the report

In [17]:
joblib.dump(model, MODEL_PATH, compress=3)
size_mb = MODEL_PATH.stat().st_size / 1e6
assert size_mb < 95, f"Model is {size_mb:.0f} MB; GitHub rejects files over 100 MB. Reduce trees or depth."

# Reload and check it gives the same answers
reloaded = joblib.load(MODEL_PATH)
assert (reloaded.predict(X_test.iloc[:5000]) == pred[:5000]).all()
assert list(reloaded.feature_names_in_) == SELECTED_FEATURES
print(f"saved {MODEL_PATH.name}: {size_mb:.1f} MB, reload check passed")

features_doc = {
    "model_version": MODEL_VERSION,
    "dataset": "CICIDS2017 (MachineLearningCVE CSVs)",
    "classes": CLASSES,
    "features": SELECTED_FEATURES,
    "units": {
        "Flow Duration": "microseconds",
        "IAT features": "microseconds between consecutive packets",
        "Length features": "TCP/UDP payload bytes, headers not included",
        "Flow Bytes/s, Flow Packets/s": "per second, over the whole flow",
    },
    "note": "Order matters. The engine must build the vector in exactly this order.",
}
(TRAIN_DIR / "features.json").write_text(json.dumps(features_doc, indent=2) + "\n", encoding="utf-8")

meta = {
    "model_version": MODEL_VERSION, "trained_on": str(date.today()), "seed": SEED,
    "scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__,
    "joblib": joblib.__version__, "python": platform.python_version(),
    "classes": CLASSES, "features": SELECTED_FEATURES,
    "train_rows": len(train), "test_rows": len(test),
    "macro_f1": report["macro avg"]["f1-score"], "accuracy": report["accuracy"],
}
(MODEL_DIR / f"model_{MODEL_VERSION}.meta.json").write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
print("saved features.json and", f"model_{MODEL_VERSION}.meta.json")

saved model_v1.pkl: 7.8 MB, reload check passed
saved features.json and model_v1.meta.json


In [18]:
def md_table(frame, floatfmt="{:.4f}"):
    frame = frame.reset_index()
    out = ["| " + " | ".join(str(c) for c in frame.columns) + " |", "|" + "---|" * len(frame.columns)]
    for _, r in frame.iterrows():
        out.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(out)

per_class = pd.DataFrame({c: report[c] for c in CLASSES}).T[["precision", "recall", "f1-score", "support"]]
per_class["support"] = per_class["support"].astype(int)
per_class.index.name = "class"
exp_cols = ["n_features", "macro_f1", "Botnet P", "Botnet R", "BruteForce R", "Benign R"]
thr = pd.DataFrame({
    "threshold": [f"{t:.2f}" for t in threshold_table.index],
    "attack flows blocked": [f"{v:.2%}" for v in threshold_table["attack flows blocked"]],
    "benign flows blocked by mistake": [f"{v:.3%}" for v in threshold_table["benign flows blocked by mistake"]],
    "benign mistakes per 100,000 flows": [f"{v:.0f}" for v in threshold_table["benign mistakes per 100,000 flows"]],
}).set_index("threshold")
clean = lambda s: re.sub(r"[^ -~]+", "-", s)

lines = [
    f"# Model report: model_{MODEL_VERSION}",
    "",
    f"Generated by `train_cicids2017.ipynb` on {date.today()}. Do not edit by hand; rerun the notebook.",
    "",
    "## Summary",
    "",
    f"- **Algorithm:** Random Forest, 100 trees, max depth 20, inside one scikit-learn pipeline",
    f"- **Dataset:** CICIDS2017, {counts['raw rows']:,} raw flows",
    f"- **Features:** {len(SELECTED_FEATURES)} (see `features.json`)",
    f"- **Classes:** {', '.join(CLASSES)}",
    f"- **Test accuracy:** {report['accuracy']:.4f}",
    f"- **Test macro F1:** {report['macro avg']['f1-score']:.4f}",
    f"- **Training time:** {train_seconds:.0f} s",
    f"- **Model file:** {size_mb:.1f} MB",
    f"- **Environment:** Python {platform.python_version()}, scikit-learn {sklearn.__version__}, seed {SEED}",
    "",
    "## Data",
    "",
    "| Step | Rows |", "|---|---|",
    *[f"| {k} | {v:,} |" for k, v in counts.items()],
    "",
    "Labels dropped: " + ", ".join(f"{clean(k)} ({v:,})" for k, v in dropped_labels.items()) + ".",
    "",
    f"Benign was downsampled to {BENIGN_SAMPLE:,} flows; all attack flows were kept.",
    "",
    md_table(split_table.rename_axis("class")),
    "",
    "## Results on the test set",
    "",
    md_table(per_class),
    "",
    "Confusion matrix (rows are the true class, columns the predicted class):",
    "",
    md_table(cm_df.rename_axis("")),
    "",
    f"![Confusion matrix]({cm_png.name})",
    "",
    "## Confidence threshold for automatic blocking",
    "",
    md_table(thr),
    "",
    "## Experiment: feature groups",
    "",
    md_table(feature_results[exp_cols]),
    "",
    "`core` was selected. Flag counts add nothing and cannot be reproduced live. Window sizes and "
    "destination port score higher but do so by recognising the lab's attacker machines and service ports.",
    "",
    "## Experiment: class imbalance",
    "",
    md_table(imbalance_results[exp_cols]),
    "",
    "No weighting was selected. Weighting raises Botnet recall but its precision collapses.",
    "",
    "## Feature importance",
    "",
    md_table(importances.rename("importance").rename_axis("feature")),
    "",
    "## Known limitations",
    "",
    f"- **Botnet is the weak class** (recall {report['Botnet']['recall']:.2f}). Its flows look like ordinary web traffic at this feature level.",
    "- **Web attacks, Infiltration and Heartbleed are not detected.** They were excluded for lack of samples.",
    "- **These scores are for this dataset.** The test set comes from the same lab network as the training set, "
    "so results on a different network will be lower. The engine therefore also uses rate-based rules and a confidence threshold.",
    "- **Live features must match.** The engine has to compute each feature the way the dataset does (see `features.json`).",
    "",
]
report_path = REPORT_DIR / f"model_{MODEL_VERSION}.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print("saved", report_path.name)

saved model_v1.md
